In [64]:
import os
import numpy as np
import pandas as pd
from scipy.stats import weibull_min
from datetime import datetime, timedelta

In [65]:
os.makedirs('data', exist_ok = True)

In [76]:
class SyntheticCohortGenerator:
    CHANNELS = {
        'organic': {
            'cac': 20,
            'arpu': 15,
            'weibull_shape':1.3,
            'weibull_scale': 25,
            'monthly_signups': 800
        
        },
        'referral': {
            'cac':25,
            'arpu':14,
            'weibull_shape':1.25,
            'weibull_scale': 23,
            'monthly_signups': 400
        },
        'paid_search' : {
            'cac':45,
            'arpu':16,
            'weibull_shape': 0.95,
            'weibull_scale': 15,
            'monthly_signups': 600
        },
        'paid_social' : {
            'cac':70,
            'arpu':15,
            'weibull_shape': 0.8,
            'weibull_scale': 12,
            'monthly_signups': 300
        },

         'direct' : {
            'cac': 35,
            'arpu': 17,
            'weibull_shape': 1.15,
            'weibull_scale': 20,
            'monthly_signups': 500
         }
    }
    
    def __init__(self, n_months= 24, start_date = '2023-01-01', discount_rate= 0.10):
         self.n_months = n_months
         self.start_date = pd.to_datetime(start_date)
         self.discount_rate = discount_rate
         self.df = None
         
    def weibull_retention(self, months_since_signup, shape, scale):
        return np.exp(-(months_since_signup/ scale) ** shape)

    def generate(self):
        rows = []
    
        for cohort_idx in range(self.n_months):
            cohort_date = self.start_date + timedelta(days = 30 * cohort_idx)
            cohort_label = cohort_date.strftime('%Y-%m')
    
            for channel_name, params in self.CHANNELS.items():
                cohort_size = params['monthly_signups']
                cac = params['cac']
                arpu = params['arpu']
                shape = params['weibull_shape']
                scale = params['weibull_scale']
    
                for months_since in range(self.n_months - cohort_idx):
                    retention_rate = self.weibull_retention(months_since, shape, scale)
                    active_users = int(cohort_size * retention_rate)
                    monthly_revenue = active_users * arpu
    
                    rows.append({
                        'cohort_month': cohort_label,
                        'cohort_date': cohort_date,
                        'channel':channel_name,
                        'cac': cac,
                        'arpu': arpu,
                        'weibull_shape': shape,
                        'weibull_scale': scale,
                        'months_since_signup': months_since,
                        'cohort_size': cohort_size,
                        'active_users': active_users,
                        'retention_rate': retention_rate,
                        'monthly_revenue':monthly_revenue
    
                    })
        self.df = pd.DataFrame(rows)
        return self.df  

    
    def calculate_ltv(self):
        ltv_data= []
    
        for(cohort, channel), group in self.df.groupby(['cohort_month', 'channel']):
            group = group.sort_values('months_since_signup')
    
            ltv = 0
            cumulative_revenue = 0
            payback_month = None
    
            first_row = group.iloc[0]
            cac = first_row['cac']
    
            for _, row in group.iterrows():
                month = row['months_since_signup']
                revenue = row['monthly_revenue']
                discounted_revenue = revenue / ((1+ self.discount_rate) ** month)
                ltv += discounted_revenue
                cumulative_revenue += revenue
    
                if payback_month is None and cumulative_revenue >= cac:
                    payback_month = month
    
            ltv_ratio = ltv / cac if cac > 0 else 0
    
            ltv_data.append({
                'cohort_month': cohort,
                'channel': channel,
                'cac': cac,
                'ltv': round(ltv, 2),
                'ltv_ratio': round(ltv, 2),
                'payback_month': payback_month if payback_month else np.nan
            })
    
        return pd.DataFrame(ltv_data)
        
    def save_to_csv(self, filename= 'synthetic_cohorts.csv'):
        if self.df is None:
            self.generate()
        self.df.to_csv(filename, index= False)
        print(f"Data saved to {filename} ({len(self.df)} rows)")
        return filename

    def summary_stats(self):
        if self.df is None:
            self.generate()
    
        print("\n" + "="*70)
        print("SYNTHETIC COHORT DATA SUMMARY")
        print("=" * 70)
    
        print(f"\nDate Range: {self.df['cohort_date'].min().date()} to {self.df['cohort_date'].max().date()}")
        print(f"Total Rows: {len(self.df):,}")
        print(f"Cohorts: {self.df['cohort_month'].nunique()}")
        print(f"Channels: {self.df['channel'].nunique()}")
        
        print("\nChannel Characteristics:")
        print("-" * 70)
        
        for channel in self.CHANNELS.keys():
            params = self.CHANNELS[channel]
            print(f"\n{channel.upper()}")
            print(f" CAC: $ {params['cac']}")
            print(f" ARPU: $ {params['arpu']}/month")
            print(f" Weibull Shape: {params['weibull_shape']} (churn type)")
            print(f" Weibull Scale: {params['weibull_scale']} (months to decay)")
            print(f" Monthly Signups:{params['monthly_signups']:,}")
    
        print("\n" + "=" * 70)
    
if __name__ == "__main__":
    generator= SyntheticCohortGenerator(n_months = 24, start_date = '2023-01-01')

    df = generator.generate()
    print(f"Generated {len(df)} rows of cohort data")

    print("Sample data (first 10 rows):")
    print(df.head(10))

    generator.save_to_csv('synthetic_cohorts.csv')

    ltv_df = generator.calculate_ltv()
    print("\nSample LTV calculations (first 10 rows):")
    print(ltv_df.head(10))
    ltv_df.to_csv('data/cohort_ltv.csv', index= False)

generator.summary_stats()   
              

Generated 1500 rows of cohort data
Sample data (first 10 rows):
  cohort_month cohort_date  channel  cac  arpu  weibull_shape  weibull_scale  \
0      2023-01  2023-01-01  organic   20    15            1.3             25   
1      2023-01  2023-01-01  organic   20    15            1.3             25   
2      2023-01  2023-01-01  organic   20    15            1.3             25   
3      2023-01  2023-01-01  organic   20    15            1.3             25   
4      2023-01  2023-01-01  organic   20    15            1.3             25   
5      2023-01  2023-01-01  organic   20    15            1.3             25   
6      2023-01  2023-01-01  organic   20    15            1.3             25   
7      2023-01  2023-01-01  organic   20    15            1.3             25   
8      2023-01  2023-01-01  organic   20    15            1.3             25   
9      2023-01  2023-01-01  organic   20    15            1.3             25   

   months_since_signup  cohort_size  active_users  rete